# 16. Michael_acceptor_1 / imine_2 데이터형 규칙 확장

## 이번 노트북에서 할 것
- Michael_acceptor_1이 기존 michael_acceptor와 중복인지 먼저 확인
  (중복이면 통합/스킵, 다른 하위패턴이면 신규 추가)
- imine_2가 imine_1(옥심)과 다른 이민 하위형인지 실제 매치 분자로 확인
- 신규 규칙은 MMPA 기반 전하+분자량 필터로 치환 후보 발굴
- replacement_library.py 반영, 회귀 테스트, held-out 커버리지 재측정

## 간략한 정리 (15까지)
- 라이브러리 11개 규칙, 전 규칙 후보 2개 이상 확보
- 3-endpoint(Tox21/Ames/hERG) 검증 체계 완성, LLM은 Ames에서 가장 뚜렷한 우위(5/7)
- QED 롤백 안전장치는 검증 결과 불필요 판단(15개 다중문제 분자 전부 QED 개선,
  하락 0건, 단계당 개선폭 ~0.09로 일정)
- 다음 확장 후보 분류 확정: Michael_acceptor_1/imine_2는 데이터형(Colab),
  catechol/thiol_2/quinone_A 등은 문헌형(학생 직접)
- Michael_acceptor_1이 빈도(30회)와 데이터형 기준 둘 다 만족하는 최우선 후보로 결정

## 다음에 해야 할 것 (오늘 끝나면)
- 문헌형 규칙(catechol 등)은 학생이 별도로 채워 넣을 예정
- quinone_A는 고리형 구조라 SMARTS/Case B 난이도 있음 - 별도 시간 배분 필요
- 전체 held-out set 최종 대량 실행 검토

In [1]:
# 셀 1
!pip install rdkit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 13.7 MB/s eta 0:00:00


In [2]:
# 셀 2
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

Cloning into 'laidd-2026'...
remote: Enumerating objects: 175, done.
remote: Counting objects: 100% (175/175), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 175 (delta 82), reused 121 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (175/175), 563.92 KiB | 5.47 MiB/s, done.
Resolving deltas: 100% (82/82), done.
/content/laidd-2026
/content/laidd-2026


In [3]:
# 셀 3
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [4]:
# 셀 4
import importlib
from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import find_core_and_target, reassemble_molecule, propose_fix, canonicalize, iterative_fix_loop
from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

data = load_tox21_clean()
print("도구 로드 확인 완료")

[06:01:12] WARNING: not removing hydrogen atom without neighbors
[06:01:12] Explicit valence for atom # 8 Al, 6, is greater than permitted
[06:01:13] Explicit valence for atom # 3 Al, 6, is greater than permitted
[06:01:13] Explicit valence for atom # 4 Al, 6, is greater than permitted
[06:01:13] Explicit valence for atom # 4 Al, 6, is greater than permitted
[06:01:13] Explicit valence for atom # 9 Al, 6, is greater than permitted
[06:01:13] Explicit valence for atom # 5 Al, 6, is greater than permitted
[06:01:14] Explicit valence for atom # 16 Al, 6, is greater than permitted
[06:01:14] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[06:01:14] WARNING: not removing hydrogen atom without neighbors


도구 로드 확인 완료


In [5]:
# michael_acceptor(기존)와 Michael_acceptor_1(신규 후보)가 실제로 다른 분자에 매치되는지 확인
overlap_check = []
for s in data['smiles_train'][:1000]:
    p = detect_toxicophores(s)
    rule_names = [x['rule_name'] for x in p]
    has_old = 'michael_acceptor' in rule_names  # 주의: 실제 이름 다를 수 있음, 확인 필요
    has_new = 'Michael_acceptor_1' in rule_names
    if has_old or has_new:
        overlap_check.append((s, has_old, has_new))

print(f"확인된 분자: {len(overlap_check)}개")
for s, old, new in overlap_check[:10]:
    print(f"old={old}, new={new}: {s[:50]}")

확인된 분자: 30개
old=False, new=True: C=CC(=O)OCCCCCC
old=False, new=True: CC(=O)C=CC1=C(C)CCCC1(C)C
old=False, new=True: COc1cc(C=CC(=O)N2CCN(CC(=O)N3CCCC3)CC2)cc(OC)c1OC
old=False, new=True: CC=CC=O
old=False, new=True: CC1(C)C[C@@H]1C(=O)N/C(=C\CCCCSC[C@H](N)C(=O)O)C(=
old=False, new=True: C=C(C)C(=O)Nc1ccc(Cl)c(Cl)c1
old=False, new=True: C=CC(C)=O
old=False, new=True: C=C(C)C(=O)OC
old=False, new=True: C[C@@H]1O[C@@H](O[C@@H]2[C@@H](O)[C@H](OCCc3ccc(O)
old=False, new=True: O=C(O)/C=C(\CC(=O)O)C(=O)O


In [6]:
mol_test = Chem.MolFromSmiles("C=CC(=O)OCCCCCC")
pattern_old = Chem.MolFromSmarts("C=CC(=O)")
print("old(michael_acceptor) 매치:", mol_test.HasSubstructMatch(pattern_old))

old(michael_acceptor) 매치: True


In [7]:
mol_test = "C=CC(=O)OCCCCCC"
result = detect_toxicophores(mol_test)
print(result)

[{'rule_name': 'Aliphatic_long_chain', 'atom_indices': [4, 5, 6, 7]}, {'rule_name': 'Michael_acceptor_1', 'atom_indices': [0, 1, 2, 3]}]


In [8]:
count_michael_old = 0
count_michael_1 = 0
for s in data['smiles_test']:
    p = detect_toxicophores(s)
    names = [x['rule_name'] for x in p]
    if 'michael_acceptor' in names:
        count_michael_old += 1
    if 'Michael_acceptor_1' in names:
        count_michael_1 += 1

print(f"'michael_acceptor'(우리가 쓰던 이름) 발견: {count_michael_old}개")
print(f"'Michael_acceptor_1'(실제 이름) 발견: {count_michael_1}개")

'michael_acceptor'(우리가 쓰던 이름) 발견: 0개
'Michael_acceptor_1'(실제 이름) 발견: 50개


In [10]:
!cat src/tools/replacement_library.py


REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "michael_acceptor": {
        "problem_smarts": "C=CC(=O)",
        "candidates": [
            {"smiles": "CCC(=O)", "name": "saturated ketone",
   

In [9]:
all_rule_names = list(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'].keys())
print("현재 라이브러리 규칙 전체:", all_rule_names)
print()

# 전체 데이터(train+valid+test)에서 각 규칙이 몇 번 실제로 발동하는지 확인
all_smiles_combined = list(data['smiles_train']) + list(data['smiles_valid']) + list(data['smiles_test'])

rule_hit_counts = {name: 0 for name in all_rule_names}

for s in all_smiles_combined:
    p = detect_toxicophores(s)
    found_names = set(x['rule_name'] for x in p)
    for rule in all_rule_names:
        if rule in found_names:
            rule_hit_counts[rule] += 1

print("=== 규칙별 실제 발동 횟수 (전체 7,823개 분자 기준) ===")
for rule, count in rule_hit_counts.items():
    status = "✅ 정상" if count > 0 else "❌ 이름 불일치 의심!"
    print(f"{rule}: {count}회  {status}")

현재 라이브러리 규칙 전체: ['nitro_group', 'aldehyde', 'michael_acceptor', 'thiourea', 'acyl_halide', 'alkyl_halide', 'aniline', 'phenol', 'amide', 'Sulfonic_acid_2', 'imine_1']



[06:01:32] WARNING: not removing hydrogen atom without neighbors


=== 규칙별 실제 발동 횟수 (전체 7,823개 분자 기준) ===
nitro_group: 343회  ✅ 정상
aldehyde: 166회  ✅ 정상
michael_acceptor: 0회  ❌ 이름 불일치 의심!
thiourea: 0회  ❌ 이름 불일치 의심!
acyl_halide: 0회  ❌ 이름 불일치 의심!
alkyl_halide: 372회  ✅ 정상
aniline: 313회  ✅ 정상
phenol: 0회  ❌ 이름 불일치 의심!
amide: 0회  ❌ 이름 불일치 의심!
Sulfonic_acid_2: 228회  ✅ 정상
imine_1: 237회  ✅ 정상


In [12]:
target_check = ["michael_acceptor", "thiourea", "acyl_halide", "phenol", "amide"]

for target in target_check:
    print(f"\n--- {target} 관련 규칙 찾기 ---")
    for s in all_smiles_combined[:3000]:
        p = detect_toxicophores(s)
        for x in p:
            if target.lower() in x['rule_name'].lower():
                print(f"  {s[:40]} -> {x['rule_name']}")
                break


--- michael_acceptor 관련 규칙 찾기 ---
  C=CC(=O)OCCCCCC -> Michael_acceptor_1
  CC(=O)C=CC1=C(C)CCCC1(C)C -> Michael_acceptor_1
  COc1cc(C=CC(=O)N2CCN(CC(=O)N3CCCC3)CC2)c -> Michael_acceptor_1
  CC=CC=O -> Michael_acceptor_1
  CC1(C)C[C@@H]1C(=O)N/C(=C\CCCCSC[C@H](N) -> Michael_acceptor_1
  C=C(C)C(=O)Nc1ccc(Cl)c(Cl)c1 -> Michael_acceptor_1
  C=CC(C)=O -> Michael_acceptor_1
  C=C(C)C(=O)OC -> Michael_acceptor_1
  C[C@@H]1O[C@@H](O[C@@H]2[C@@H](O)[C@H](O -> Michael_acceptor_1
  O=C(O)/C=C(\CC(=O)O)C(=O)O -> Michael_acceptor_1
  CC(/C=C/C=C(\C)C(=O)O[C@@H]1O[C@H](CO[C@ -> Michael_acceptor_1
  CC=CC(=O)CCCC -> Michael_acceptor_1
  C/C=C(\C)C(=O)OCCc1ccccc1 -> Michael_acceptor_1
  C=C(C)C(=O)OCC(O)CO -> Michael_acceptor_1
  C=C(C)C(=O)OCCOc1ccccc1 -> Michael_acceptor_1
  CC1=C(/C=C/C(C)=C/C=C\C(C)=C\C(=O)Nc2ccc -> Michael_acceptor_1
  C=C(C)C(=O)OCCCCCCCCCCCCCCCCCC -> Michael_acceptor_1
  C=CC(=O)OCCO -> Michael_acceptor_1
  CCCC=C(C=O)CC -> Michael_acceptor_1
  CCOC(=O)C(C#N)=C(c1ccccc1)c1cc

[02:56:53] WARNING: not removing hydrogen atom without neighbors


  CCCCC(O)/C=C(C)/C=C/C=C/C(=O)N1CCCC1=O -> Michael_acceptor_1

--- thiourea 관련 규칙 찾기 ---


[02:57:02] WARNING: not removing hydrogen atom without neighbors



--- acyl_halide 관련 규칙 찾기 ---


[02:57:13] WARNING: not removing hydrogen atom without neighbors



--- phenol 관련 규칙 찾기 ---
  CC(=O)Nc1ccc(OC(C)=O)cc1 -> phenol_ester
  CCCCCCCC(=O)Oc1c(Br)cc(C#N)cc1Br -> phenol_ester
  C/C=C(C(=C/C)/c1ccc(OC(C)=O)cc1)\c1ccc(O -> phenol_ester
  O=C1CCc2ccccc2O1 -> phenol_ester
  CC(=O)Nc1ccc(OC(=O)c2ccccc2O)cc1 -> phenol_ester
  CN(C)C(=O)COC(=O)Cc1ccc(OC(=O)c2ccc(NC(= -> phenol_ester
  Nc1ccc(C(=O)Oc2ccccc2)c(O)c1 -> phenol_ester
  O=C(OC[C@H]1O[C@@H](OC(=O)c2cc(O)c(O)c(O -> phenol_ester
  CC1=C(/C=C/C(C)=C/C=C/C(C)=C/C(=O)Oc2c(C -> phenol_ester
  CNCC(O)c1ccc(OC(=O)C(C)(C)C)c(OC(=O)C(C) -> phenol_ester
  CC(=O)Oc1ccc(C(c2ccc(OC(C)=O)cc2)c2ccccn -> phenol_ester
  Cc1ccc(OC(=O)c2ccccc2O)cc1 -> phenol_ester
  CCC(=O)Oc1ccc2c(c1)CC[C@@H]1[C@@H]2CC[C@ -> phenol_ester
  Cc1ccc(C(=O)Oc2ccc(C(O)CNC(C)(C)C)cc2OC( -> phenol_ester
  O=C(Oc1ccccc1)c1cccc(C(=O)Oc2ccccc2)c1 -> phenol_ester
  C/C=C/C(=O)Oc1c(CCCCCC(C)C)cc([N+](=O)[O -> phenol_ester
  CN(C(=O)C(Cl)Cl)c1ccc(OC(=O)c2ccco2)cc1 -> phenol_ester
  COCCOC[C@H](CC1(C(=O)N[C@H]2CC[C@@H](C(= -> phenol_este

[02:57:23] WARNING: not removing hydrogen atom without neighbors



--- amide 관련 규칙 찾기 ---
  N#CCN(CC#N)CCN(CC#N)CC#N -> cyanamide
  N#CCNCC#N -> cyanamide


[02:57:33] WARNING: not removing hydrogen atom without neighbors


In [10]:
all_catalog_names = set()
for s in all_smiles_combined[:3000]:
    p = detect_toxicophores(s)
    for x in p:
        all_catalog_names.add(x['rule_name'])

keywords = ['thio', 'urea', 'acyl', 'halide', 'amide', 'phenol', 'hydroxyl', 'OH']
for kw in keywords:
    matches = [n for n in all_catalog_names if kw.lower() in n.lower()]
    print(f"'{kw}' 포함 규칙: {matches}")

'thio' 포함 규칙: ['thiol_1', 'Thiocarbonyl_group', 'thiophene_amino_Aa(45)', 'het_thio_666_A(13)', 'cyanate_/aminonitrile_/thiocyanate', 'thiol_2', 'thioester']
'urea' 포함 규칙: []
'acyl' 포함 규칙: ['acyl_hydrazine']
'halide' 포함 규칙: ['alkyl_halide', 'acid_halide']
'amide' 포함 규칙: ['cyanamide']
'phenol' 포함 규칙: ['hzone_phenol_A(479)', 'phenol_ester', 'phenol_sulfite_A(1)']
'hydroxyl' 포함 규칙: ['N-hydroxyl_pyridine']
'OH' 포함 규칙: ['cyanohydrins']


[06:02:04] WARNING: not removing hydrogen atom without neighbors


In [11]:
%%writefile src/tools/replacement_library.py

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "Michael_acceptor_1": {
        "problem_smarts": "C=CC(=O)",
        "candidates": [
            {"smiles": "CCC(=O)", "name": "saturated ketone",
             "rationale": "이중결합을 제거해 단백질 친전자성 부가반응(covalent binding) 위험 제거"},
        ],
    },
    "acid_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
            {"smiles": "C(=O)O", "name": "ester",
             "rationale": "아마이드보다 극성이 낮고 유연한 대체 옵션, 가수분해 속도 조절 가능 (검증 필요)"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "problem_smarts": "[NH2]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "acetamide (acylated amine)",
             "rationale": "1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "반응성 아민을 제거하면서 전자끄는기로 고리 전자밀도 보정"},
        ],
    },
    "Sulfonic_acid_2": {
        "problem_smarts": "S(=O)(=O)[OX2H1,OX1-]",
        "candidates": [
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "생리적 pH에서 이온화 정도(전하)를 크게 낮춰 세포막 투과성을 "
                          "개선함. 설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 "
                          "저해되는 경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 "
                          "중성에 가까워 약물유사성이 개선됨"},
            {"smiles": "C(=O)O", "name": "carboxylic acid",
             "rationale": "설폰산보다 산성도가 약하고 부피가 작은 산성 bioisostere "
                          "(검증 필요)"},
        ],
    },
    "imine_1": {
        "problem_smarts": "C=N[OX2H1]",
        "candidates": [
            {"smiles": "CN", "name": "amine (reduced)",
             "rationale": "옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 "
                          "카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 "
                          "경로를 제거함"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "옥심의 탈수 반응으로 니트릴을 얻는 것은 잘 알려진 화학 변환, "
                          "극성을 낮추면서 대사 불안정성 개선 (검증 필요)"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)

Overwriting src/tools/replacement_library.py


In [13]:
import importlib
import src.tools.replacement_library
import src.tools.molecule_editor

importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.molecule_editor)
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import propose_fix

all_rule_names_v2 = list(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'].keys())
rule_hit_counts_v2 = {name: 0 for name in all_rule_names_v2}

for s in all_smiles_combined:
    p = detect_toxicophores(s)
    found_names = set(x['rule_name'] for x in p)
    for rule in all_rule_names_v2:
        if rule in found_names:
            rule_hit_counts_v2[rule] += 1

print("=== 수정 후 규칙별 실제 발동 횟수 ===")
for rule, count in rule_hit_counts_v2.items():
    status = "✅" if count > 0 else "❌ 여전히 문제"
    print(f"{rule}: {count}회  {status}")

[06:05:23] WARNING: not removing hydrogen atom without neighbors


=== 수정 후 규칙별 실제 발동 횟수 ===
nitro_group: 343회  ✅
aldehyde: 166회  ✅
Michael_acceptor_1: 291회  ✅
acid_halide: 34회  ✅
alkyl_halide: 372회  ✅
aniline: 313회  ✅
Sulfonic_acid_2: 228회  ✅
imine_1: 237회  ✅


In [14]:
!git add src/tools/replacement_library.py
!git commit -m "CRITICAL FIX: remove 3 dead rules (thiourea, phenol, amide) that had no corresponding FilterCatalog entry and never fired in automated pipeline despite passing manual tests; fix naming for michael_acceptor->Michael_acceptor_1 and acyl_halide->acid_halide. Verified all 8 remaining rules fire correctly across full dataset (7823 molecules)."
!git push https://{token}@github.com/Dec32th/laidd-2026.git

[main 827f509] CRITICAL FIX: remove 3 dead rules (thiourea, phenol, amide) that had no corresponding FilterCatalog entry and never fired in automated pipeline despite passing manual tests; fix naming for michael_acceptor->Michael_acceptor_1 and acyl_halide->acid_halide. Verified all 8 remaining rules fire correctly across full dataset (7823 molecules).
 1 file changed, 2 insertions(+), 33 deletions(-)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 690 bytes | 690.00 KiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   67b4e58..827f509  main -> main


In [15]:
# 아까 검색에서 이미 확인된 것들 재정리
candidates_confirmed = {
    "catechol": None,  # 아직 확인 안 함
    "thiol_1": "발견됨",  # 아까 'thio' 검색에서 나옴
    "thiol_2": "발견됨",  # 아까 'thio' 검색에서 나옴
    "Thiocarbonyl_group": "발견됨",
    "acyl_hydrazine": "발견됨",  # 아까 'acyl' 검색에서 나옴
}

# 전체 카탈로그 이름 목록에서 다음 후보들 빈도 확인
target_new = ["thiol_1", "thiol_2", "Thiocarbonyl_group", "acyl_hydrazine", "catechol",
              "N-hydroxyl_pyridine", "cyanohydrins"]

for rule in target_new:
    count = sum(1 for s in all_smiles_combined if rule in [x['rule_name'] for x in detect_toxicophores(s)])
    print(f"{rule}: {count}회")

[06:08:52] WARNING: not removing hydrogen atom without neighbors


thiol_1: 21회


[06:09:16] WARNING: not removing hydrogen atom without neighbors


thiol_2: 57회


[06:09:40] WARNING: not removing hydrogen atom without neighbors


...

catechol: 82회


[06:10:55] WARNING: not removing hydrogen atom without neighbors


N-hydroxyl_pyridine: 2회


[06:11:20] WARNING: not removing hydrogen atom without neighbors


cyanohydrins: 3회


In [16]:
examples_new = {}
for s in all_smiles_combined[:3000]:
    problems = detect_toxicophores(s)
    for p in problems:
        if p['rule_name'] in ["Thiocarbonyl_group", "catechol"] and p['rule_name'] not in examples_new:
            examples_new[p['rule_name']] = (s, p['atom_indices'])
    if len(examples_new) == 2:
        break

for name, (smi, indices) in examples_new.items():
    print(f"\n{name}: {smi}")
    mol = Chem.MolFromSmiles(smi)
    for idx in indices:
        atom = mol.GetAtomWithIdx(idx)
        print(f"  인덱스 {idx}: {atom.GetSymbol()} (이웃: {[n.GetSymbol() for n in atom.GetNeighbors()]})")


Thiocarbonyl_group: CC(C)(C)c1n[nH]c(=S)n(N)c1=O
  인덱스 7: C (이웃: ['N', 'S', 'N'])
  인덱스 8: S (이웃: ['C'])

catechol: N#CC(C#N)=Cc1ccc(O)c(O)c1
  인덱스 6: C (이웃: ['C', 'C', 'C'])
  인덱스 7: C (이웃: ['C', 'C'])
  인덱스 8: C (이웃: ['C', 'C'])
  인덱스 9: C (이웃: ['C', 'O', 'C'])
  인덱스 10: O (이웃: ['C'])
  인덱스 11: C (이웃: ['C', 'O', 'C'])
  인덱스 12: O (이웃: ['C'])
  인덱스 13: C (이웃: ['C', 'C'])


In [17]:
test_thio = "CC(C)(C)c1n[nH]c(=S)n(N)c1=O"
pattern_thio = Chem.MolFromSmarts("C=S")
print("pattern size:", pattern_thio.GetNumAtoms())

mol_thio = Chem.MolFromSmiles(test_thio)
print("매치 여부:", mol_thio.HasSubstructMatch(pattern_thio))

pattern size: 2
매치 여부: False


In [18]:
mol_thio2 = Chem.MolFromSmiles(test_thio)
for atom in mol_thio2.GetAtoms():
    if atom.GetSymbol() == 'S':
        print(f"S 원자 인덱스 {atom.GetIdx()}: 이웃 {[(n.GetSymbol(), mol_thio2.GetBondBetweenAtoms(atom.GetIdx(), n.GetIdx()).GetBondTypeAsDouble()) for n in atom.GetNeighbors()]}")

S 원자 인덱스 8: 이웃 [('C', 2.0)]


In [19]:
pattern_thio2 = Chem.MolFromSmarts("[#6]=[#16]")
print("일반 결합 매치:", mol_thio2.HasSubstructMatch(pattern_thio2))

pattern_thio3 = Chem.MolFromSmarts("[#6]~[#16]")  # ~는 결합 종류 무관하게 매치
print("결합 종류 무관 매치:", mol_thio2.HasSubstructMatch(pattern_thio3))

일반 결합 매치: True
결합 종류 무관 매치: True


In [20]:
%%writefile src/tools/replacement_library.py

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "Michael_acceptor_1": {
        "problem_smarts": "C=CC(=O)",
        "candidates": [
            {"smiles": "CCC(=O)", "name": "saturated ketone",
             "rationale": "이중결합을 제거해 단백질 친전자성 부가반응(covalent binding) 위험 제거"},
        ],
    },
    "acid_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
            {"smiles": "C(=O)O", "name": "ester",
             "rationale": "아마이드보다 극성이 낮고 유연한 대체 옵션, 가수분해 속도 조절 가능 (검증 필요)"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "problem_smarts": "[NH2]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "acetamide (acylated amine)",
             "rationale": "1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "반응성 아민을 제거하면서 전자끄는기로 고리 전자밀도 보정"},
        ],
    },
    "Sulfonic_acid_2": {
        "problem_smarts": "S(=O)(=O)[OX2H1,OX1-]",
        "candidates": [
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "생리적 pH에서 이온화 정도(전하)를 크게 낮춰 세포막 투과성을 "
                          "개선함. 설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 "
                          "저해되는 경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 "
                          "중성에 가까워 약물유사성이 개선됨"},
            {"smiles": "C(=O)O", "name": "carboxylic acid",
             "rationale": "설폰산보다 산성도가 약하고 부피가 작은 산성 bioisostere "
                          "(검증 필요)"},
        ],
    },
    "imine_1": {
        "problem_smarts": "C=N[OX2H1]",
        "candidates": [
            {"smiles": "CN", "name": "amine (reduced)",
             "rationale": "옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 "
                          "카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 "
                          "경로를 제거함"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "옥심의 탈수 반응으로 니트릴을 얻는 것은 잘 알려진 화학 변환, "
                          "극성을 낮추면서 대사 불안정성 개선 (검증 필요)"},
        ],
    },
    "Thiocarbonyl_group": {
        "problem_smarts": "[#6]=[#16]",
        "candidates": [
            {"smiles": "N", "name": "urea-like (O replacing S)",
             "rationale": "황을 산소로 대체(티오카르보닐->카르보닐)하는 것은 흔한 "
                          "bioisostere 전략으로, 갑상선 기능 저해 등 황 함유 작용기 "
                          "특유의 대사/독성 우려를 낮춤 (검증 필요, thiourea->urea "
                          "치환 논리와 동일 계열)"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)

Overwriting src/tools/replacement_library.py


In [21]:
%%writefile src/tools/replacement_library.py

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "Michael_acceptor_1": {
        "problem_smarts": "C=CC(=O)",
        "candidates": [
            {"smiles": "CCC(=O)", "name": "saturated ketone",
             "rationale": "이중결합을 제거해 단백질 친전자성 부가반응(covalent binding) 위험 제거"},
        ],
    },
    "acid_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
            {"smiles": "C(=O)O", "name": "ester",
             "rationale": "아마이드보다 극성이 낮고 유연한 대체 옵션, 가수분해 속도 조절 가능 (검증 필요)"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "problem_smarts": "[NH2]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "acetamide (acylated amine)",
             "rationale": "1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "반응성 아민을 제거하면서 전자끄는기로 고리 전자밀도 보정"},
        ],
    },
    "Sulfonic_acid_2": {
        "problem_smarts": "S(=O)(=O)[OX2H1,OX1-]",
        "candidates": [
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "생리적 pH에서 이온화 정도(전하)를 크게 낮춰 세포막 투과성을 "
                          "개선함. 설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 "
                          "저해되는 경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 "
                          "중성에 가까워 약물유사성이 개선됨"},
            {"smiles": "C(=O)O", "name": "carboxylic acid",
             "rationale": "설폰산보다 산성도가 약하고 부피가 작은 산성 bioisostere "
                          "(검증 필요)"},
        ],
    },
    "imine_1": {
        "problem_smarts": "C=N[OX2H1]",
        "candidates": [
            {"smiles": "CN", "name": "amine (reduced)",
             "rationale": "옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 "
                          "카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 "
                          "경로를 제거함"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "옥심의 탈수 반응으로 니트릴을 얻는 것은 잘 알려진 화학 변환, "
                          "극성을 낮추면서 대사 불안정성 개선 (검증 필요)"},
        ],
    },
    "Thiocarbonyl_group": {
        "problem_smarts": "[#6]=[#16]",
        "candidates": [
            {"smiles": "O", "name": "carbonyl (O replacing S)",
             "rationale": "황을 산소로 대체(티오카르보닐->카르보닐)하는 것은 흔한 "
                          "bioisostere 전략으로, 갑상선 기능 저해 등 황 함유 작용기 "
                          "특유의 대사/독성 우려를 낮춤 (검증 필요, thiourea->urea "
                          "치환 논리와 동일 계열)"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)

Overwriting src/tools/replacement_library.py


In [22]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import find_core_and_target, propose_fix

test_thio = "CC(C)(C)c1n[nH]c(=S)n(N)c1=O"
print("Thiocarbonyl core_test:", find_core_and_target(test_thio, "Thiocarbonyl_group"))
print("Thiocarbonyl fix_test:", propose_fix(test_thio, "Thiocarbonyl_group", candidate_idx=0))

# 회귀 테스트
print("\n=== 회귀 테스트 ===")
print(propose_fix("O=C(O)CCl", "alkyl_halide", candidate_idx=0))
print(propose_fix("O=C(Cl)C(Cl)Cl", "acid_halide", candidate_idx=0))

Thiocarbonyl core_test: None
Thiocarbonyl fix_test: None

=== 회귀 테스트 ===
{'new_smiles': 'O=C(O)CO', 'candidate_used': 'hydroxyl (alcohol)', 'rationale': '이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지', 'is_valid': True}
{'new_smiles': 'NC(=O)C(Cl)Cl', 'candidate_used': 'amide', 'rationale': '고반응성 아실할라이드를 안정적인 아마이드로 대체', 'is_valid': True}


In [23]:
mol_check_thio = Chem.MolFromSmiles(test_thio)

fragments1_thio = rdMMPA.FragmentMol(mol_check_thio, maxCuts=1, resultsAsMols=False)
print("=== maxCuts=1 ===")
for core, chain in fragments1_thio:
    print(f"core: {core}, chain: {chain}")

fragments2_thio = rdMMPA.FragmentMol(mol_check_thio, maxCuts=2, resultsAsMols=False)
print("\n=== maxCuts=2 ===")
for core, chain in fragments2_thio:
    print(f"core: {core}, chain: {chain}")

=== maxCuts=1 ===
core: , chain: CC(C)(c1n[nH]c(=S)n(N)c1=O)[*:1].C[*:1]
core: , chain: CC(C)(C)[*:1].Nn1c(=S)[nH]nc([*:1])c1=O

=== maxCuts=2 ===
core: , chain: CC(C)(c1n[nH]c(=S)n(N)c1=O)[*:1].C[*:1]
core: CC(c1n[nH]c(=S)n(N)c1=O)([*:1])[*:2], chain: C[*:1].C[*:2]
core: CC(C)([*:1])[*:2], chain: C[*:1].Nn1c(=S)[nH]nc([*:2])c1=O
core: , chain: CC(C)(C)[*:1].Nn1c(=S)[nH]nc([*:1])c1=O


In [24]:
test_catechol = "N#CC(C#N)=Cc1ccc(O)c(O)c1"
print("catechol core_test:", find_core_and_target(test_catechol, "catechol"))

catechol core_test: None


In [25]:
pattern_catechol = Chem.MolFromSmarts("[OX2H]c(c)[OX2H]")
# 또는 더 정확하게 두 OH가 인접 탄소에 있다는 걸 명시
pattern_catechol2 = Chem.MolFromSmarts("c(O)c(O)")

mol_catechol = Chem.MolFromSmiles(test_catechol)
print("pattern1 매치:", mol_catechol.HasSubstructMatch(pattern_catechol) if pattern_catechol else "파싱실패")
print("pattern2 매치:", mol_catechol.HasSubstructMatch(pattern_catechol2) if pattern_catechol2 else "파싱실패")

if pattern_catechol2:
    print("pattern2 크기:", pattern_catechol2.GetNumAtoms())

pattern1 매치: False
pattern2 매치: True
pattern2 크기: 4


In [26]:
pattern_catechol3 = Chem.MolFromSmarts("[OX2H]cc[OX2H]")
print("catechol3 (산소 2개 모두 패턴에 포함):", mol_catechol.HasSubstructMatch(pattern_catechol3))
print("catechol3 크기:", pattern_catechol3.GetNumAtoms() if pattern_catechol3 else None)

catechol3 (산소 2개 모두 패턴에 포함): True
catechol3 크기: 4


In [27]:
pattern_catechol4 = Chem.MolFromSmarts("[OX2H;$(Oc1ccccc1O)]")
print("catechol4 매치:", mol_catechol.HasSubstructMatch(pattern_catechol4) if pattern_catechol4 else "파싱실패")
print("catechol4 크기:", pattern_catechol4.GetNumAtoms() if pattern_catechol4 else None)

catechol4 매치: True
catechol4 크기: 1


In [29]:
%%writefile src/tools/replacement_library.py

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "Michael_acceptor_1": {
        "problem_smarts": "C=CC(=O)",
        "candidates": [
            {"smiles": "CCC(=O)", "name": "saturated ketone",
             "rationale": "이중결합을 제거해 단백질 친전자성 부가반응(covalent binding) 위험 제거"},
        ],
    },
    "acid_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
            {"smiles": "C(=O)O", "name": "ester",
             "rationale": "아마이드보다 극성이 낮고 유연한 대체 옵션, 가수분해 속도 조절 가능 (검증 필요)"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "problem_smarts": "[NH2]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "acetamide (acylated amine)",
             "rationale": "1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "반응성 아민을 제거하면서 전자끄는기로 고리 전자밀도 보정"},
        ],
    },
    "Sulfonic_acid_2": {
        "problem_smarts": "S(=O)(=O)[OX2H1,OX1-]",
        "candidates": [
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "생리적 pH에서 이온화 정도(전하)를 크게 낮춰 세포막 투과성을 "
                          "개선함. 설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 "
                          "저해되는 경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 "
                          "중성에 가까워 약물유사성이 개선됨"},
            {"smiles": "C(=O)O", "name": "carboxylic acid",
             "rationale": "설폰산보다 산성도가 약하고 부피가 작은 산성 bioisostere "
                          "(검증 필요)"},
        ],
    },
    "imine_1": {
        "problem_smarts": "C=N[OX2H1]",
        "candidates": [
            {"smiles": "CN", "name": "amine (reduced)",
             "rationale": "옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 "
                          "카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 "
                          "경로를 제거함"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "옥심의 탈수 반응으로 니트릴을 얻는 것은 잘 알려진 화학 변환, "
                          "극성을 낮추면서 대사 불안정성 개선 (검증 필요)"},
        ],
    },
    "catechol": {
        "problem_smarts": "[OX2H;$(Oc1ccccc1O)]",
        "candidates": [
            {"smiles": "C", "name": "methoxy",
             "rationale": "인체의 COMT(catechol-O-methyltransferase) 효소가 카테콜을 "
                          "메톡시페놀로 메틸화하여 해독하는 생리적 경로와 동일한 원리. "
                          "오르토-퀴논으로의 산화 경로를 차단하여 세포독성/유전독성 우려를 "
                          "낮춤 (학생 확인 예정: ScienceDirect catechol overview, "
                          "PMC6643002 등 참고)"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)

Overwriting src/tools/replacement_library.py


In [30]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import find_core_and_target, propose_fix

test_catechol = "N#CC(C#N)=Cc1ccc(O)c(O)c1"
print("catechol core_test:", find_core_and_target(test_catechol, "catechol"))
print("catechol fix_test:", propose_fix(test_catechol, "catechol", candidate_idx=0))

print("\n=== 회귀 테스트 ===")
print(propose_fix("O=C(O)CCl", "alkyl_halide", candidate_idx=0))
print(propose_fix(test_thio if 'test_thio' in dir() else "CC(C)(C)c1n[nH]c(=S)n(N)c1=O", "Sulfonic_acid_2", candidate_idx=0) if False else None)
print(propose_fix("Nc1ccc(NCCO)c([N+](=O)[O-])c1", "aniline", candidate_idx=0))

catechol core_test: None
catechol fix_test: None

=== 회귀 테스트 ===
{'new_smiles': 'O=C(O)CO', 'candidate_used': 'hydroxyl (alcohol)', 'rationale': '이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지', 'is_valid': True}
None
{'new_smiles': 'NC(=O)c1ccc(NCCO)c([N+](=O)[O-])c1', 'candidate_used': 'acetamide (acylated amine)', 'rationale': '1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단', 'is_valid': True}


In [31]:
# 실제로 조각이 어떻게 나오는지 직접 봅시다
fragments1_catechol = rdMMPA.FragmentMol(Chem.MolFromSmiles(test_catechol), maxCuts=1, resultsAsMols=False)
for core, chain in fragments1_catechol:
    print(f"core: {core}, chain: {chain}")

core: , chain: N#CC(=Cc1ccc(O)c(O)c1)[*:1].N#C[*:1]
core: , chain: N#CC(C#N)=C[*:1].Oc1ccc([*:1])cc1O
core: , chain: N#CC(C#N)=Cc1ccc([*:1])c(O)c1.O[*:1]
core: , chain: N#CC(C#N)=Cc1ccc(O)c([*:1])c1.O[*:1]


In [32]:
%%writefile src/tools/atom_editor.py
from rdkit import Chem


def apply_atom_edit(smiles: str, smarts: str, target_idx_in_pattern: int,
                     edit_type: str, edit_param):
    """분자를 조각내지 않고, SMARTS로 찾은 특정 원자를 직접 편집.
    edit_type='replace_element': edit_param=새 원자번호(int), 원소만 교체
    edit_type='add_substituent': edit_param=붙일 조각의 SMILES(str), H 하나를 떼고 결합"""
    mol = Chem.MolFromSmiles(smiles)
    pattern = Chem.MolFromSmarts(smarts)
    if mol is None or pattern is None:
        return None

    matches = mol.GetSubstructMatches(pattern)
    if not matches:
        return None
    target_idx = matches[0][target_idx_in_pattern]

    rwmol = Chem.RWMol(mol)

    if edit_type == "replace_element":
        atom = rwmol.GetAtomWithIdx(target_idx)
        atom.SetAtomicNum(edit_param)

    elif edit_type == "add_substituent":
        frag = Chem.MolFromSmiles(edit_param)
        if frag is None:
            return None
        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol = Chem.RWMol(combined)
        offset = mol.GetNumAtoms()  # 원본 분자 원자 개수 = 붙인 조각의 시작 인덱스
        rwmol.AddBond(target_idx, offset, Chem.BondType.SINGLE)
        atom = rwmol.GetAtomWithIdx(target_idx)
        if atom.GetNumExplicitHs() > 0:
            atom.SetNumExplicitHs(atom.GetNumExplicitHs() - 1)
        else:
            atom.SetNoImplicit(False)
    else:
        return None

    try:
        new_mol = rwmol.GetMol()
        Chem.SanitizeMol(new_mol)
    except Exception as e:
        print("Sanitize 실패:", e)
        return None

    return Chem.MolToSmiles(new_mol)

Writing src/tools/atom_editor.py


In [33]:
import importlib
import src.tools.atom_editor
importlib.reload(src.tools.atom_editor)
from src.tools.atom_editor import apply_atom_edit

test_catechol = "N#CC(C#N)=Cc1ccc(O)c(O)c1"
result_catechol = apply_atom_edit(
    test_catechol,
    smarts="[OX2H;$(Oc1ccccc1O)]",
    target_idx_in_pattern=0,
    edit_type="add_substituent",
    edit_param="C"
)
print("catechol 결과:", result_catechol)
print("유효성:", Chem.MolFromSmiles(result_catechol) is not None if result_catechol else "실패")

catechol 결과: COc1ccc(C=C(C#N)C#N)cc1O
유효성: True


In [34]:
test_thio = "CC(C)(C)c1n[nH]c(=S)n(N)c1=O"
result_thio = apply_atom_edit(
    test_thio,
    smarts="[#6]=[#16]",
    target_idx_in_pattern=1,  # 패턴의 두 번째 원자(S)를 타겟
    edit_type="replace_element",
    edit_param=8  # 산소의 원자번호
)
print("Thiocarbonyl 결과:", result_thio)
print("유효성:", Chem.MolFromSmiles(result_thio) is not None if result_thio else "실패")

Thiocarbonyl 결과: CC(C)(C)c1n[nH]c(=O)n(N)c1=O
유효성: True


In [35]:
%%writefile src/tools/replacement_library.py

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "Michael_acceptor_1": {
        "problem_smarts": "C=CC(=O)",
        "candidates": [
            {"smiles": "CCC(=O)", "name": "saturated ketone",
             "rationale": "이중결합을 제거해 단백질 친전자성 부가반응(covalent binding) 위험 제거"},
        ],
    },
    "acid_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
            {"smiles": "C(=O)O", "name": "ester",
             "rationale": "아마이드보다 극성이 낮고 유연한 대체 옵션, 가수분해 속도 조절 가능 (검증 필요)"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "problem_smarts": "[NH2]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "acetamide (acylated amine)",
             "rationale": "1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "반응성 아민을 제거하면서 전자끄는기로 고리 전자밀도 보정"},
        ],
    },
    "Sulfonic_acid_2": {
        "problem_smarts": "S(=O)(=O)[OX2H1,OX1-]",
        "candidates": [
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "생리적 pH에서 이온화 정도(전하)를 크게 낮춰 세포막 투과성을 "
                          "개선함. 설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 "
                          "저해되는 경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 "
                          "중성에 가까워 약물유사성이 개선됨"},
            {"smiles": "C(=O)O", "name": "carboxylic acid",
             "rationale": "설폰산보다 산성도가 약하고 부피가 작은 산성 bioisostere "
                          "(검증 필요)"},
        ],
    },
    "imine_1": {
        "problem_smarts": "C=N[OX2H1]",
        "candidates": [
            {"smiles": "CN", "name": "amine (reduced)",
             "rationale": "옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 "
                          "카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 "
                          "경로를 제거함"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "옥심의 탈수 반응으로 니트릴을 얻는 것은 잘 알려진 화학 변환, "
                          "극성을 낮추면서 대사 불안정성 개선 (검증 필요)"},
        ],
    },
    "catechol": {
        "edit_method": "atom_edit",
        "problem_smarts": "[OX2H;$(Oc1ccccc1O)]",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "add_substituent", "param": "C", "name": "methoxy",
             "rationale": "인체의 COMT(catechol-O-methyltransferase) 효소가 카테콜을 "
                          "메톡시페놀로 메틸화하여 해독하는 생리적 경로와 동일한 원리. "
                          "오르토-퀴논으로의 산화 경로를 차단하여 세포독성/유전독성 우려를 "
                          "낮춤 (학생 확인 예정: ScienceDirect catechol overview, "
                          "PMC6643002 등 참고)"},
        ],
    },
    "Thiocarbonyl_group": {
        "edit_method": "atom_edit",
        "problem_smarts": "[#6]=[#16]",
        "target_idx_in_pattern": 1,
        "candidates": [
            {"edit_type": "replace_element", "param": 8, "name": "carbonyl (O replacing S)",
             "rationale": "황을 산소로 대체(티오카르보닐->카르보닐)하는 것은 흔한 "
                          "bioisostere 전략으로, 갑상선 기능 저해 등 황 함유 작용기 "
                          "특유의 대사/독성 우려를 낮춤 (검증 필요, thiourea->urea "
                          "치환 논리와 동일 계열)"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)

Overwriting src/tools/replacement_library.py


In [36]:
%%writefile src/tools/atom_editor.py
from rdkit import Chem


def apply_atom_edit_from_rule(smiles: str, rule_name: str, candidate_idx: int = 0):
    """replacement_library의 atom_edit 규칙을 이용해 원자 직접 편집을 수행."""
    from src.tools.replacement_library import get_replacement_candidates
    info = get_replacement_candidates(rule_name)
    if info is None or info.get("edit_method") != "atom_edit":
        return None
    if candidate_idx >= len(info["candidates"]):
        return None

    candidate = info["candidates"][candidate_idx]
    smarts = info["problem_smarts"]
    target_idx_in_pattern = info["target_idx_in_pattern"]

    mol = Chem.MolFromSmiles(smiles)
    pattern = Chem.MolFromSmarts(smarts)
    if mol is None or pattern is None:
        return None

    matches = mol.GetSubstructMatches(pattern)
    if not matches:
        return None
    target_idx = matches[0][target_idx_in_pattern]

    rwmol = Chem.RWMol(mol)

    if candidate["edit_type"] == "replace_element":
        atom = rwmol.GetAtomWithIdx(target_idx)
        atom.SetAtomicNum(candidate["param"])

    elif candidate["edit_type"] == "add_substituent":
        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None
        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol = Chem.RWMol(combined)
        offset = mol.GetNumAtoms()
        rwmol.AddBond(target_idx, offset, Chem.BondType.SINGLE)
        atom = rwmol.GetAtomWithIdx(target_idx)
        if atom.GetNumExplicitHs() > 0:
            atom.SetNumExplicitHs(atom.GetNumExplicitHs() - 1)
        else:
            atom.SetNoImplicit(False)
    else:
        return None

    try:
        new_mol = rwmol.GetMol()
        Chem.SanitizeMol(new_mol)
    except Exception:
        return None

    new_smiles = Chem.MolToSmiles(new_mol)
    is_valid = Chem.MolFromSmiles(new_smiles) is not None

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate["name"],
        "rationale": candidate["rationale"],
        "is_valid": is_valid,
    }

Overwriting src/tools/atom_editor.py


In [38]:
%%writefile src/tools/molecule_editor.py
from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.replacement_library import get_replacement_candidates


def _check_and_match(part_smiles, problem_pattern, pattern_size):
    """조각이 problem_pattern과 정확한 크기로 매치되는지 확인."""
    part_mol = Chem.MolFromSmiles(part_smiles.replace('[*:1]', 'C').replace('[*:2]', 'C'))
    if part_mol is None or not part_mol.HasSubstructMatch(problem_pattern):
        return False
    n_attachment = part_smiles.count('[*:')
    return part_mol.GetNumHeavyAtoms() - n_attachment == pattern_size


def find_core_and_target(smiles: str, rule_name: str):
    """분자에서 rule_name에 해당하는 문제구조를 담은 조각(target)과
    나머지 뼈대(core)를 찾아서 반환.
    1단계(maxCuts=1)로 단순 분리를 먼저 시도하고,
    실패하면 2단계(maxCuts=2)로 고리 인접 작용기 분리를 시도한다."""
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    problem_pattern = Chem.MolFromSmarts(info['problem_smarts'])
    pattern_size = problem_pattern.GetNumAtoms()

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # --- Case A: 단순 2조각 분리 (maxCuts=1) ---
    fragments1 = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)
    for core, chain in fragments1:
        if core:
            continue
        parts = chain.split('.')
        if len(parts) != 2:
            continue
        for i, part in enumerate(parts):
            if _check_and_match(part, problem_pattern, pattern_size):
                return {"core": parts[1 - i], "target_removed": part}

    # --- Case B: 고리 인접 등, core가 남는 2-cut 분리 ---
    fragments2 = rdMMPA.FragmentMol(mol, maxCuts=2, resultsAsMols=False)
    for core, chain in fragments2:
        if not core:
            continue
        chain_parts = chain.split('.')
        if len(chain_parts) != 2:
            continue
        for i, part in enumerate(chain_parts):
            if not _check_and_match(part, problem_pattern, pattern_size):
                continue
            other_chain_part = chain_parts[1 - i]

            target_ap = '[*:1]' if '[*:1]' in part else ('[*:2]' if '[*:2]' in part else None)
            if target_ap is None:
                continue

            core_mol = Chem.MolFromSmiles(core)
            other_mol = Chem.MolFromSmiles(other_chain_part)
            if core_mol is None or other_mol is None:
                continue
            try:
                merged = Chem.molzip(core_mol, other_mol)
            except Exception:
                continue

            merged_smiles = Chem.MolToSmiles(merged)
            if merged_smiles.count('[*:') != 1:
                continue
            if '[*:1]' not in merged_smiles:
                merged_smiles = merged_smiles.replace('[*:2]', '[*:1]')

            return {"core": merged_smiles, "target_removed": part}

    return None


def reassemble_molecule(core_smiles: str, rule_name: str, candidate_idx: int = 0):
    info = get_replacement_candidates(rule_name)
    if info is None or candidate_idx >= len(info['candidates']):
        return None
    candidate = info['candidates'][candidate_idx]

    core_mol = Chem.MolFromSmiles(core_smiles)
    replacement_mol = Chem.MolFromSmiles(f"[*:1]{candidate['smiles']}")
    if core_mol is None or replacement_mol is None:
        return None

    try:
        combined = Chem.molzip(core_mol, replacement_mol)
        new_smiles = Chem.MolToSmiles(combined)
    except Exception:
        return None

    is_valid = Chem.MolFromSmiles(new_smiles) is not None

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate['name'],
        "rationale": candidate['rationale'],
        "is_valid": is_valid,
    }


def propose_fix(smiles: str, rule_name: str, candidate_idx: int = 0):
    """규칙의 edit_method에 따라 결합절단형(기존) 또는 원자직접편집형(신규)으로 분기."""
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    if info.get("edit_method") == "atom_edit":
        from src.tools.atom_editor import apply_atom_edit_from_rule
        return apply_atom_edit_from_rule(smiles, rule_name, candidate_idx)

    located = find_core_and_target(smiles, rule_name)
    if located is None:
        return None
    return reassemble_molecule(located['core'], rule_name, candidate_idx)


def canonicalize(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol) if mol else None


def iterative_fix_loop(smiles: str, max_iterations: int = 10, candidate_idx: int = 0,
                        llm_client=None, llm_model=None, llm_client_type="gemini"):
    """진단->치환->재평가를 반복.
    llm_client가 주어지면: 어떤 문제부터 고칠지 + 어떤 후보를 쓸지 둘 다 LLM이 판단.
    llm_client_type: "gemini" 또는 "openai_compatible".
    llm_client가 없으면: 리스트 순서(known_problems[0]) + candidate_idx 고정값 사용."""
    from src.tools.toxicophore_detector import detect_toxicophores
    from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

    current = canonicalize(smiles)
    seen = {current}
    history = [{"step": 0, "smiles": current}]
    skipped_rules = []

    for step in range(1, max_iterations + 1):
        problems = detect_toxicophores(current)
        history[-1]["problems"] = problems

        if not problems:
            return {"status": "success", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        known_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]
        unknown_problems = [p for p in problems if p not in known_problems]

        for p in unknown_problems:
            if p['rule_name'] not in skipped_rules:
                skipped_rules.append(p['rule_name'])

        if not known_problems:
            return {"status": "no_known_fix", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        if llm_client is not None:
            problem_decision = ask_llm_which_problem_to_fix(llm_client, llm_model, current, problems, client_type=llm_client_type)
            target_rule = problem_decision['rule_name']
            problem_reason = problem_decision.get('reason', '')

            candidate_decision = ask_llm_which_candidate_to_use(llm_client, llm_model, current, target_rule, client_type=llm_client_type)
            chosen_candidate_idx = candidate_decision['candidate_idx']
            candidate_reason = candidate_decision.get('reason', '')
        else:
            target_rule = known_problems[0]['rule_name']
            problem_reason = "규칙 기반(리스트 순서대로)"
            chosen_candidate_idx = candidate_idx
            candidate_reason = "규칙 기반(고정 인덱스)"

        fixed = propose_fix(current, target_rule, chosen_candidate_idx)

        if fixed is None or not fixed['is_valid']:
            return {"status": "stuck", "reason": f"'{target_rule}' 치환 실패", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        new_current = canonicalize(fixed['new_smiles'])

        if new_current in seen:
            return {"status": "cycle_detected", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        seen.add(new_current)
        current = new_current
        history.append({
            "step": step,
            "smiles": current,
            "fixed_rule": target_rule,
            "problem_reason": problem_reason,
            "candidate_used": fixed['candidate_used'],
            "candidate_reason": candidate_reason,
        })

    return {"status": "max_iterations_reached", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

Overwriting src/tools/molecule_editor.py


In [39]:
import importlib
import src.tools.replacement_library
import src.tools.atom_editor
import src.tools.molecule_editor
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix

print("=== 회귀 테스트 (기존 결합절단형) ===")
print(propose_fix("O=C(O)CCl", "alkyl_halide", candidate_idx=0))
print(propose_fix("Nc1ccc(NCCO)c([N+](=O)[O-])c1", "aniline", candidate_idx=0))

print("\n=== 신규 (원자편집형) ===")
print(propose_fix("N#CC(C#N)=Cc1ccc(O)c(O)c1", "catechol", candidate_idx=0))
print(propose_fix("CC(C)(C)c1n[nH]c(=S)n(N)c1=O", "Thiocarbonyl_group", candidate_idx=0))

=== 회귀 테스트 (기존 결합절단형) ===
{'new_smiles': 'O=C(O)CO', 'candidate_used': 'hydroxyl (alcohol)', 'rationale': '이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지', 'is_valid': True}
{'new_smiles': 'NC(=O)c1ccc(NCCO)c([N+](=O)[O-])c1', 'candidate_used': 'acetamide (acylated amine)', 'rationale': '1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단', 'is_valid': True}

=== 신규 (원자편집형) ===
{'new_smiles': 'COc1ccc(C=C(C#N)C#N)cc1O', 'candidate_used': 'methoxy', 'rationale': '인체의 COMT(catechol-O-methyltransferase) 효소가 카테콜을 메톡시페놀로 메틸화하여 해독하는 생리적 경로와 동일한 원리. 오르토-퀴논으로의 산화 경로를 차단하여 세포독성/유전독성 우려를 낮춤 (학생 확인 예정: ScienceDirect catechol overview, PMC6643002 등 참고)', 'is_valid': True}
{'new_smiles': 'CC(C)(C)c1n[nH]c(=O)n(N)c1=O', 'candidate_used': 'carbonyl (O replacing S)', 'rationale': '황을 산소로 대체(티오카르보닐->카르보닐)하는 것은 흔한 bioisostere 전략으로, 갑상선 기능 저해 등 황 함유 작용기 특유의 대사/독성 우려를 낮춤 (검증 필요, thiourea->urea 치환 논리와 동일 계열)', 'is_valid': True}


In [40]:
all_rule_names_v3 = list(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'].keys())
print("현재 라이브러리:", all_rule_names_v3)

rule_hit_counts_v3 = {name: 0 for name in all_rule_names_v3}
for s in all_smiles_combined:
    p = detect_toxicophores(s)
    found_names = set(x['rule_name'] for x in p)
    for rule in all_rule_names_v3:
        if rule in found_names:
            rule_hit_counts_v3[rule] += 1

print("\n=== 전체 라이브러리(10개) 발동 횟수 ===")
for rule, count in rule_hit_counts_v3.items():
    status = "✅" if count > 0 else "❌"
    print(f"{rule}: {count}회  {status}")

# iterative_fix_loop 실전 테스트
print("\n=== iterative_fix_loop 실전 (catechol 포함 분자) ===")
result_catechol_loop = iterative_fix_loop("N#CC(C#N)=Cc1ccc(O)c(O)c1", max_iterations=5)
print("상태:", result_catechol_loop['status'])
for h in result_catechol_loop['history']:
    print(h)

현재 라이브러리: ['nitro_group', 'aldehyde', 'Michael_acceptor_1', 'acid_halide', 'alkyl_halide', 'aniline', 'Sulfonic_acid_2', 'imine_1', 'catechol', 'Thiocarbonyl_group']


[06:39:07] WARNING: not removing hydrogen atom without neighbors



=== 전체 라이브러리(10개) 발동 횟수 ===
nitro_group: 343회  ✅
aldehyde: 166회  ✅
Michael_acceptor_1: 291회  ✅
acid_halide: 34회  ✅
alkyl_halide: 372회  ✅
aniline: 313회  ✅
Sulfonic_acid_2: 228회  ✅
imine_1: 237회  ✅
catechol: 82회  ✅
Thiocarbonyl_group: 88회  ✅

=== iterative_fix_loop 실전 (catechol 포함 분자) ===
상태: no_known_fix
{'step': 0, 'smiles': 'N#CC(C#N)=Cc1ccc(O)c(O)c1', 'problems': [{'rule_name': 'catechol_A(92)', 'atom_indices': [6, 7, 8, 9, 10, 11, 12, 13]}, {'rule_name': 'ene_cyano_A(19)', 'atom_indices': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 13]}, {'rule_name': 'catechol', 'atom_indices': [6, 7, 8, 9, 10, 11, 12, 13]}, {'rule_name': 'conjugated_nitrile_group', 'atom_indices': [0, 1, 2, 5]}]}
{'step': 1, 'smiles': 'COc1ccc(C=C(C#N)C#N)cc1O', 'fixed_rule': 'catechol', 'problem_reason': '규칙 기반(리스트 순서대로)', 'candidate_used': 'methoxy', 'candidate_reason': '규칙 기반(고정 인덱스)', 'problems': [{'rule_name': 'ene_cyano_A(19)', 'atom_indices': [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]}, {'rule_name': 'conjugated_nitr

In [41]:
!git add src/tools/replacement_library.py src/tools/atom_editor.py src/tools/molecule_editor.py
!git status

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   src/tools/atom_editor.py
	modified:   src/tools/molecule_editor.py
	modified:   src/tools/replacement_library.py



In [42]:
!git commit -m "Add atom-level edit path (RWMol-based) for rules that can't be handled by fragment-cut approach: catechol (context-dependent recursive SMARTS, add_substituent) and Thiocarbonyl_group (ring-embedded atom, replace_element). propose_fix now branches by edit_method. All 10 rules verified firing across full dataset (7823 molecules)."
!git push https://{token}@github.com/Dec32th/laidd-2026.git

[main 4be53ca] Add atom-level edit path (RWMol-based) for rules that can't be handled by fragment-cut approach: catechol (context-dependent recursive SMARTS, add_substituent) and Thiocarbonyl_group (ring-embedded atom, replace_element). propose_fix now branches by edit_method. All 10 rules verified firing across full dataset (7823 molecules).
 3 files changed, 100 insertions(+), 7 deletions(-)
 create mode 100644 src/tools/atom_editor.py
Enumerating objects: 12, done.
Counting objects: 100% (12/12), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 2.47 KiB | 631.00 KiB/s, done.
Total 7 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/Dec32th/laidd-2026.git
   827f509..4be53ca  main -> main
